# Figure 1

The nearest words of three queries in five models, the country-to-capital offsets of DINOv3-H and DINOv3-7B, and six country-capital pairs of DINOv3-H projected on a plane.

Inputs: `scripts/analysis/figure1_neighbours.py`, which writes `results/neighbours.parquet`, `results/relation_offsets.parquet` and `results/figure1b_projection.parquet`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb
from matplotlib.patches import FancyArrowPatch

from src.models import PRETTY
from src.paper import style, tables
from src.paper.results import read

style.setup_notebook()

neighbours = read("neighbours")
offsets = read("relation_offsets").set_index("model")
projection = read("figure1b_projection")

## Panel (a)

The three nearest words of each query in each of the five models, among all other words, names and pseudowords.

In [ ]:
panel_a = tables.figure1a_neighbours(neighbours)
panel_a.write()
panel_a.frame

## Consistent offsets

In DINOv3-H and DINOv3-7B, each country is queried alone and with the mean country-to-capital offset of the other folds added. Offset alignment is the mean cosine between each held-out offset and that mean, and its null re-pairs countries with capitals.

In [ ]:
OFFSET_MODELS = ("dinov3-h", "dinov3-7b")
for model in OFFSET_MODELS:
    row = offsets.loc[model]
    print(f"{PRETTY[model]}: capital ranked first for {row.first_without_offset} of "
          f"{row.n_pairs} countries from the country alone, {row.first_with_offset} "
          f"with the mean offset of the other folds added")
for model in OFFSET_MODELS:
    row = offsets.loc[model]
    print(f"{PRETTY[model]}: offset alignment {row.alignment_mean:.3f} against a null mean "
          f"of {row.alignment_mean_null_mean:.3f}, p = {row.alignment_mean_null_p:.3f}")

## Panel (b)

The word directions are scaled to unit length, and the endpoints of the displayed country-capital pairs are centered on their mean. They are projected on the mean offset of the undisplayed pairs and on the leading principal component of the displayed midpoints orthogonal to that offset.

In [ ]:
diagnostic = projection[projection.record_type.eq("diagnostic")].iloc[0]
coordinates = projection[projection.record_type.eq("coordinate")].reset_index(drop=True)
n_display, n_undisplayed = int(diagnostic.n_display), int(diagnostic.n_undisplayed)
print(f"pairs drawn: {n_display} of {n_display + n_undisplayed}, in {PRETTY[diagnostic.model]}")
coordinates[["source", "target", "endpoint", "word", "x", "y"]]

Every point keeps its place in the plane up to a horizontal stretch, and each label takes the first position clear of the other marks.

In [ ]:
#: The canvas, which sets the size of the panel's minipage in the paper.
WIDTH_IN = 2.7775
HEIGHT_PT = 156.0370
#: The box where the paper sets the (b) letter, which no label may enter:
#: left, bottom, right, top, in points from the lower left corner.
LETTER_BOX_PT = (0.0, HEIGHT_PT - 11.0, 14.0, HEIGHT_PT)
#: The frame the endpoints are fitted into and the origin of the rays, in points
#: from the lower left corner. The vertical axis has no sign of its own, so
#: ``vertical_sign`` chooses which way up the pairs read. The horizontal scale is
#: ``x_stretch`` times the vertical one, capped by the room.
LAYOUT = {
    "origin": (9.0, 7.0),
    "left": 46.0,
    "right": 154.0,
    "bottom": 16.0,
    "top": 146.0,
    "vertical_sign": -1,
    "x_stretch": 1.6,
}
#: The country's ray is dashed and darker, the capital's dotted and paler. Each
#: tone is its ink blended that far towards white.
RAY_TONE = {"source": 0.30, "target": 0.18}
RAY_DASH = {"source": (0, (3.2, 2.0)), "target": (0, (0.6, 1.9))}
RAY_WIDTH = 0.85
#: The offset arrows, which stop short of both endpoints.
RELATION_STYLE = {"tail_width": 1.1, "head_width": 4.4, "head_length": 5.2}
RELATION_SHRINK_PT = (2.5, 3.2)
#: Endpoint marker area in square points, 4 pt across.
ENDPOINT_SIZE = 16.0
#: Where the axis arrowheads end, in points from the lower left corner.
AXIS_END_PT = (WIDTH_IN * 72 - 3.0, HEIGHT_PT - 3.0)
AXIS_COLOUR = style.INK
#: The white halo behind every label, where a ray passes under it.
HALO_WIDTH_PT = 3.2
#: The stroke that sets each label's weight, since the installed face renders a
#: bold request at regular weight.
WEIGHT_PT = {"target": 0.42, "source": 0.34, "delta": 0.32}

In [ ]:
#: Label positions in the order tried: offset in points and alignment. Countries
#: prefer the left of their point and capitals the right.
COUNTRY_CANDIDATES = (
    (-4.0, 0.0, "right", "center_baseline"), (-3.0, -3.0, "right", "top"),
    (-3.0, 3.0, "right", "bottom"), (0.0, -4.0, "center", "top"),
    (0.0, 4.0, "center", "bottom"),
)
CAPITAL_CANDIDATES = (
    (4.0, 0.0, "left", "center_baseline"), (3.0, 3.0, "left", "bottom"),
    (3.0, -3.0, "left", "top"), (0.0, 4.0, "center", "bottom"),
    (0.0, -4.0, "center", "top"),
)
#: Delta label positions: the fraction along the arrow and the side, +1 above
#: and -1 below. Above comes first, because the rays pass below every arrow.
DELTA_CANDIDATES = ((0.50, 1), (0.38, 1), (0.62, 1), (0.26, 1), (0.74, 1),
                    (0.50, -1), (0.38, -1), (0.62, -1), (0.26, -1),
                    (0.74, -1))
DELTA_NORMAL_PT = 2.2
#: Clearance kept between a label and anything else, in points.
CLEARANCE_PT = 0.6
SUBSCRIPTS = "₁₂₃₄₅₆₇₈₉"


def place(frame):
    """The endpoints in points on the canvas, and the horizontal and vertical scales.

    The vertical scale fills the height. The horizontal scale is ``x_stretch``
    times it, or as much of that as the width allows.
    """
    points = frame[["x", "y"]].to_numpy(float).copy()
    points[:, 1] *= LAYOUT["vertical_sign"]
    low, high = points.min(axis=0), points.max(axis=0)
    room = np.array([LAYOUT["right"] - LAYOUT["left"],
                     LAYOUT["top"] - LAYOUT["bottom"]])
    span = np.maximum(high - low, 1e-9)
    scale = np.full(2, float(np.min(room / span)))
    scale[0] = min(scale[0] * LAYOUT["x_stretch"], room[0] / span[0])
    placed = np.array([LAYOUT["left"], LAYOUT["bottom"]]) + (points - low) * scale
    frame = (frame[["source", "endpoint", "word"]].reset_index(drop=True)
             .assign(x=placed[:, 0], y=placed[:, 1]))
    return frame, (float(scale[0]), float(scale[1]))


def sources_of(frame):
    """Every pair in the frame, numbered from the top of the panel down."""
    return tuple(frame[frame.endpoint.eq("source")]
                 .sort_values("y", ascending=False).source)


def segment_points(start, end, samples=64):
    return start + np.linspace(0, 1, samples)[:, None] * (end - start)


def box_hits_segment(box, start, end, samples=64):
    points = segment_points(start, end, samples)
    return bool(((points[:, 0] >= box.x0) & (points[:, 0] <= box.x1)
                 & (points[:, 1] >= box.y0) & (points[:, 1] <= box.y1)).any())


def box_distance(box, points):
    """Distance from a box to the nearest of ``points``, zero inside it."""
    points = np.atleast_2d(points)
    dx = np.maximum.reduce([box.x0 - points[:, 0], np.zeros(len(points)),
                            points[:, 0] - box.x1])
    dy = np.maximum.reduce([box.y0 - points[:, 1], np.zeros(len(points)),
                            points[:, 1] - box.y1])
    return float(np.hypot(dx, dy).min())


class Placer:
    """First-fit placement of labels against the panel's marks.

    A label takes the first candidate position that clears every placed label,
    endpoint and offset arrow, the letter box and the canvas edge, and that is
    nearer its own mark than any other. It also clears the rays and axes where
    any position does.
    """

    def __init__(self, ax, frame, origin):
        self.ax = ax
        fig = ax.figure
        fig.canvas.draw()
        self.renderer = fig.canvas.get_renderer()
        points_to_px = fig.dpi / 72
        self.pad = CLEARANCE_PT * points_to_px
        to_px = ax.transData.transform
        self.endpoints = to_px(frame[["x", "y"]].to_numpy(float))
        self.radius = np.sqrt(ENDPOINT_SIZE) / 2 * points_to_px
        self.segments = []
        for source in sources_of(frame):
            pair = frame[frame.source.eq(source)].set_index("endpoint")
            self.segments.append(to_px(
                pair.loc[["source", "target"], ["x", "y"]].to_numpy(float)))
        corner = to_px(origin)
        self.rays = [np.vstack([corner, point]) for point in self.endpoints]
        self.rays += [np.vstack([corner, to_px((AXIS_END_PT[0], origin[1]))]),
                      np.vstack([corner, to_px((origin[0], AXIS_END_PT[1]))])]
        self.letter = [value * points_to_px for value in LETTER_BOX_PT]
        self.size = (fig.bbox.width, fig.bbox.height)
        self.placed = []

    def clear(self, box, rays):
        grown = box.padded(self.pad)
        if rays and any(box_hits_segment(grown, *ray) for ray in self.rays):
            return False
        width, height = self.size
        if (grown.x0 < 0 or grown.y0 < 0
                or grown.x1 > width or grown.y1 > height):
            return False
        _, bottom, right, _ = self.letter
        if grown.x0 < right and grown.y1 > bottom:
            return False
        if any(grown.overlaps(other) for other in self.placed):
            return False
        for x, y in self.endpoints:
            if (grown.x0 - self.radius < x < grown.x1 + self.radius
                    and grown.y0 - self.radius < y < grown.y1 + self.radius):
                return False
        return not any(box_hits_segment(grown, *segment)
                       for segment in self.segments)

    def nearest_own(self, box, point=None, segment=None):
        """Whether the box is nearer its own mark than any other of its kind."""
        if point is not None:
            own = self.ax.transData.transform(point)
            distances = [box_distance(box, other) for other in self.endpoints
                         if np.hypot(*(other - own)) > 1e-6]
            return box_distance(box, own) < min(distances)
        distances = [box_distance(box, segment_points(*other))
                     for other in self.segments]
        own = distances.pop(segment)
        return own < min(distances)

    def place(self, name, attempts, **own):
        """Keep the first artist from ``attempts`` that is clear and nearest its own mark."""
        for rays in (True, False):
            for make in attempts:
                artist = make()
                box = artist.get_window_extent(self.renderer)
                if self.clear(box, rays) and self.nearest_own(box, **own):
                    self.placed.append(box)
                    return artist
                artist.remove()
        raise RuntimeError(f"no clear position for the label {name}")


def word_attempts(ax, row):
    capital = row.endpoint == "target"
    candidates = CAPITAL_CANDIDATES if capital else COUNTRY_CANDIDATES
    kwargs = {
        "textcoords": "offset points",
        "fontsize": (style.FIGURE1B_CAPITAL_SIZE if capital
                     else style.FIGURE1B_COUNTRY_SIZE),
        "fontweight": "normal",
        "color": style.INK if capital else style.CONTROL,
        "annotation_clip": False, "zorder": 7,
    }
    return [
        (lambda dx=dx, dy=dy, ha=ha, va=va: ax.annotate(
            row.word, (row.x, row.y), xytext=(dx, dy), ha=ha, va=va,
            **kwargs))
        for dx, dy, ha, va in candidates]


def delta_attempts(ax, start, end, label):
    difference = end - start
    angle = np.arctan2(difference[1], difference[0])
    normal = np.array([-np.sin(angle), np.cos(angle)])
    if normal[1] < 0:
        normal *= -1
    attempts = []
    for fraction, side in DELTA_CANDIDATES:
        anchor = start + fraction * difference
        away = side * normal
        # The label's corner nearest the arrow sits at the offset, so a steep
        # arrow does not run through the label's box.
        ha = ("center" if abs(away[0]) < 0.35
              else "right" if away[0] < 0 else "left")
        va = ("center_baseline" if abs(away[1]) < 0.35
              else "bottom" if away[1] > 0 else "top")
        attempts.append(lambda anchor=anchor, away=away, ha=ha, va=va:
                        ax.annotate(
                            label, anchor, xytext=away * DELTA_NORMAL_PT,
                            textcoords="offset points", ha=ha, va=va,
                            fontsize=6.5, fontweight="normal",
                            color=style.SEMANTIC, annotation_clip=False,
                            zorder=6))
    return attempts

In [ ]:
def blend(colour, weight):
    """``colour`` at ``weight`` over white, as one opaque color."""
    rgb = np.array(to_rgb(colour))
    return tuple(weight * rgb + (1 - weight) * np.ones(3))


def halo(ax, artist):
    """A white copy under ``artist``, stroked so the rays stop short of it."""
    copy = ax.annotate(
        artist.get_text(), artist.xy, xytext=artist.xyann,
        textcoords="offset points", ha=artist.get_ha(), va=artist.get_va(),
        fontsize=artist.get_fontsize(), fontweight="normal", color=style.WHITE,
        annotation_clip=False, zorder=2)
    style.halo_text(copy, linewidth=HALO_WIDTH_PT)


def draw_scaffold(ax, origin):
    x_end, y_end = AXIS_END_PT
    axis_style = {
        "arrowstyle": "-|>",
        "mutation_scale": style.FIGURE1B_AXIS["head_scale"],
        "lw": style.FIGURE1B_AXIS["width"],
        "color": AXIS_COLOUR,
        "shrinkA": 0,
        "shrinkB": 0,
    }
    ax.annotate("", xy=(x_end, origin[1]), xytext=origin,
                arrowprops=axis_style, annotation_clip=False, zorder=0.5)
    ax.annotate("", xy=(origin[0], y_end), xytext=origin,
                arrowprops=axis_style, annotation_clip=False, zorder=0.5)


def draw_panel(ax, frame, origin):
    order = sources_of(frame)
    sources = frame[frame.endpoint.eq("source")].set_index("source")
    targets = frame[frame.endpoint.eq("target")].set_index("source")
    draw_scaffold(ax, origin)

    for source in order:
        for row in (sources.loc[source], targets.loc[source]):
            ax.plot([origin[0], row.x], [origin[1], row.y],
                    color=blend(style.INK if row.endpoint == "source" else style.CONTROL,
                                RAY_TONE[row.endpoint]),
                    lw=RAY_WIDTH, ls=RAY_DASH[row.endpoint],
                    dash_capstyle="round", zorder=1)

    for source in order:
        start, end = sources.loc[source], targets.loc[source]
        ax.add_patch(FancyArrowPatch(
            (start.x, start.y), (end.x, end.y),
            arrowstyle=(
                f"Simple,tail_width={RELATION_STYLE['tail_width']},"
                f"head_width={RELATION_STYLE['head_width']},"
                f"head_length={RELATION_STYLE['head_length']}"),
            mutation_scale=1.0, linewidth=0,
            facecolor=style.SEMANTIC, edgecolor=style.SEMANTIC,
            shrinkA=RELATION_SHRINK_PT[0], shrinkB=RELATION_SHRINK_PT[1],
            zorder=3))

    ax.scatter(sources.x, sources.y, s=ENDPOINT_SIZE,
               facecolor=style.WHITE, edgecolor=style.CONTROL,
               linewidth=0.70, zorder=4)
    ax.scatter(targets.x, targets.y, s=ENDPOINT_SIZE,
               facecolor=style.INK, edgecolor=style.INK, linewidth=0.50, zorder=4)
    ax.scatter(*origin, s=style.FIGURE1B_ORIGIN_SIZE,
               facecolor=AXIS_COLOUR, edgecolor="none", zorder=4)

    ax.set_xlim(0, WIDTH_IN * 72)
    ax.set_ylim(0, HEIGHT_PT)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")

    # Capitals are placed first, then countries, then the deltas.
    placer = Placer(ax, frame, origin)
    ordered = ([targets.loc[source] for source in order]
               + [sources.loc[source] for source in order])
    for row in ordered:
        artist = placer.place(row.word, word_attempts(ax, row),
                              point=(row.x, row.y))
        halo(ax, artist)
        style.emphasize_text(
            artist, colour=style.INK if row.endpoint == "target" else style.CONTROL,
            linewidth=WEIGHT_PT[row.endpoint])
    for number, source in enumerate(order):
        start = sources.loc[source, ["x", "y"]].to_numpy(float)
        end = targets.loc[source, ["x", "y"]].to_numpy(float)
        label = "δ" + SUBSCRIPTS[number]
        artist = placer.place(label, delta_attempts(ax, start, end, label),
                              segment=number)
        halo(ax, artist)
        style.emphasize_text(artist, colour=style.SEMANTIC,
                             linewidth=WEIGHT_PT["delta"])


def draw_figure(placed):
    fig = plt.figure(figsize=style.inches(HEIGHT_PT, WIDTH_IN))
    # One data unit is one point, so the layout can be read off the canvas.
    ax = fig.add_axes((0, 0, 1, 1))
    draw_panel(ax, placed, np.array(LAYOUT["origin"]))
    return fig

In [ ]:
placed, scale = place(coordinates)
print(f"horizontal scale: {scale[0] / scale[1]:.1f} times the vertical")
fig = draw_figure(placed)
style.save(fig, "figure1b_pairs")
plt.show()